# 08 — Single-model summary

Single-model recap: spike stats, example trials, stimulus responses, subspaces, clamped latents, fixed points, attractor and behavior perturbations

In [ ]:
import pickle
import socket
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display

np.random.seed(0)
torch.manual_seed(0)

import sys

sys.path.insert(0, str(Path.cwd().parent))

from fig_utils.task_data import (
    attach_model_session_ids,
    get_task_session,
    subset_task_params_sessions,
)
from fig_utils.transformed_rnn import transformed_rnn
from fig_utils.decoding import eval_gen_decoder, logistic_from_eff_weights
from fig_utils.perturbation import (
    compute_perturbation_distance_stats,
    generate_w_perturb_x,
    perturbation_goal_bounds,
    position_latent_indices,
    latent_mean,
    precompute_perturb_geometry,
    admissible_targets,
)
from fig_utils.plots import (
    plot_basis_2d_subspaces,
    plot_decision_distributions,
    plot_distance_moved_vs_class_mean,
    plot_perturbation_latent_snapshots,
    plot_perturbation_latent_snapshots_attractor,
    plot_time_latents,
    plot_inference_rates_vs_spikes,
    plot_example_input,
    plot_example_spikes,
    plot_example_latents,
    plot_stimulus_response_unit,
    plot_spike_stats_sessions,
    position_slice_at_time,
    position_slice_at_time,
    plot_basis_3d_trajectories,
    plot_frozen_latents,
    plot_fixed_points_subspaces,
)
from vi_rnn.inference import filtering_posterior_bootstrap
from fig_utils.spike_stats import (
    eval_spike_stats,
    session_obs_inds,
    session_stimulus_delay_activity,
    stimulus_trial_lists,
)
from vi_rnn.datasets import SWM_dataset_multi
from vi_rnn.data_utils import make_all_trials
from vi_rnn.generate import generate
from vi_rnn.saving import CPU_Unpickler, load_model

%matplotlib inline







In [ ]:
hostname = socket.gethostname()
print("hostname:", hostname)

if hostname == "MatthijsDesktop":
    out_dir = Path("/home/matthijs/swm_rnn/final_models/macaque")
    data_root = "/home/matthijs/swm_rnn/data/"
else:
    out_dir = Path("/Users/matthijs/swm_rnn_cl/final_models/macaque")
    data_root = str(Path.cwd().parent / "data") + "/"

model_name = "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_28_T_05_34_01"
# model_name = "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_30_36"
model_name = "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_26_38"
model_dir = out_dir / model_name
print("model_dir:", model_dir)

# Sessions 1 and 15 in the paper (0-based indices 0 and 14)
sess_a = 0
sess_b = 14

In [ ]:
df_basii = pickle.load(open("../data/processed/df_basii.pkl", "rb"))
df_decoding = pickle.load(open("../data/processed/df_decoding.pkl", "rb"))

row_b = df_basii.loc[df_basii["name"] == model_name]
row_d = df_decoding.loc[df_decoding["name"] == model_name]

assert not row_b.empty, "model not found in df_basii"
assert not row_d.empty, "model not found in df_decoding"

vae, training_params, task_params = load_model(
    str(model_dir), load_encoder=True, backward_compat=False
)

# Paper panels use sessions 1 and 15 only (model indices sess_a, sess_b).
summary_session_ids = [sess_a, sess_b]
task_params_summary = subset_task_params_sessions(task_params, summary_session_ids)
task_params_summary["path"] = data_root
task = attach_model_session_ids(
    SWM_dataset_multi(task_params_summary), summary_session_ids
)
print("Loaded model sessions:", summary_session_ids)
print("n_sessions in task:", len(task.sessions))

In [ ]:
# --- controls ---
# --- task / latent layout (same as 07) ---
n_pos = 3
n_pcs_time = 2
n_stim = 6

trial_dur = 5.5
n_duplications = 10

bins_before = 4
bins_after = 1
t_plot = 65
t_snapshot = 60

pos = 0
perturb_at_t = 50
noise_scale = 1.0
goal_amp = 1.0

sweep_stimulation_modes = {"regular": 0, "optogen": 10}
demo_rng_seed = 0

cmap = mpl.colors.ListedColormap(sns.color_palette("husl", n_colors=n_stim))

# fixed points / clamped latents (see notebook 04)
t_decode = 65
fixed_inds = np.arange(n_pcs_time)
freeze_time_step = 60
freeze_noise = 0.1
n_ts_plot_frozen = 140

# --- run ---
run = True

In [ ]:
u, _, labels, delay_ends = make_all_trials(
    task_params,
    dur=trial_dur,
    n_stim=n_stim,
    n_pos=n_pos,
    cue_dur=None,
    bin_size=task_params["bin_size"],
    interval_dur="mean",
    delay_dur="mean",
)


u_gen = np.concatenate([u] * n_duplications, axis=0)
labels_gen = np.concatenate([labels] * n_duplications, axis=0)
delay_ends_gen = np.concatenate([delay_ends] * n_duplications, axis=0)

u_perm = u
labels_perm = labels
delay_ends_perm = delay_ends
delay_end_ref = int(np.round(delay_ends.mean()))

print(f"u_gen: {u_gen.shape[0]} trials ({n_duplications}x duplicated)")
print(f"u_perm: {u_perm.shape[0]} unique sequences")
print(f"delay_end_ref (mean delay end, bins): {delay_end_ref}")

u_fp, _, labels_fp, _ = make_all_trials(
    task_params,
    dur=7,
    n_stim=n_stim,
    n_pos=n_pos,
    cue_dur=-1,
    bin_size=task_params["bin_size"],
    interval_dur="mean",
    delay_dur="mean",
)
u_fp = np.concatenate([u_fp] * n_duplications, axis=0)
labels_fp = np.concatenate([labels_fp] * n_duplications, axis=0)
print(f"u_fp: {u_fp.shape[0]} trials, T={u_fp.shape[2]} bins")
print(f"labels_fp shape: {labels_fp.shape}")

In [ ]:
A_comb_np = row_b["A_comb_np"].values[0]
b_comb_np = row_b["b_comb_np"].values[0]
rnn_orth = transformed_rnn(vae, A_comb_np, b_comb_np)

w_eff = row_d["w_eff"].values[0]
b_eff = row_d["b_eff"].values[0]
response_times = row_d["mean_response_onsets"].values[0]
print("response_times:", response_times)
shared_model = logistic_from_eff_weights(w_eff, b_eff, n_classes=n_stim)

z_gen = rnn_orth.simulate(u_gen)
acc_gen = eval_gen_decoder(
    z_gen,
    labels_gen,
    delay_ends_gen,
    shared_model,
    response_times,
    n_pos,
    bins_before,
    bins_after,
)
print("Gen decoder accuracy:", acc_gen)

## 1) Spike stats (sessions 1 and 15)




In [ ]:
min_data_points_isi = 5
spike_session_list = [sess_a, sess_b]

all_stats, spike_per_session = eval_spike_stats(
    vae,
    task,
    min_data_points_isi=min_data_points_isi,
    eval_train=False,
    session_list=spike_session_list,
    noise_scale=noise_scale,
    return_per_session=True,
)

summary = pd.DataFrame(
    {
        "session": [s + 1 for s in spike_session_list],
        "mean_rate": all_stats["mean_rate"],
        "std_ISI": all_stats["std_ISI"],
        "r2_pwcorr": all_stats["r2_pwcorr"],
    }
)
display(summary)

In [ ]:
plot_spike_stats_sessions(
    spike_per_session,
    save_path="../paper_figures/spikestats.pdf",
    dpi=300,
    show=True,
    box_w=0.6,
    box_h=0.6,
    panel_gap_x=0.3,
    panel_gap_y=0.15,
);

## 2) Example trial (observed vs generated)


In [ ]:
# Example trial for paper figures (session 1)
example_sess_id = sess_a
example_rng = np.random.default_rng(0)

obs_inds = session_obs_inds(vae, example_sess_id)
sess = get_task_session(task, example_sess_id)
x = sess.data_eval
u = sess.stim_eval
m = sess.loss_mask_eval
labels_ex = sess.labels_eval.cpu().numpy()

encoder_padding = training_params["encoder_padding"]
k = training_params["k"]
ed_ratio = training_params["ed_ratio"]
n_trials_ex = min(10, x.shape[0])


Z, v, data_gen, rates_gen = generate(
    vae,
    u=u[:n_trials_ex],
    x=x[:n_trials_ex],
    initial_state="prior_mean",
    k=k,
    sess_id=example_sess_id,
    noise_scale=noise_scale,
)

loss_mask = (
    m[:n_trials_ex, :-encoder_padding] if encoder_padding > 0 else m[:n_trials_ex]
)

log_likelihood, Qzs2, alphas = filtering_posterior_bootstrap(
    vae,
    x[:n_trials_ex],
    u[:n_trials_ex],
    k=k,
    sess_id=example_sess_id,
)
# print("filter log-lik (masked mean):", log_likelihood[loss_mask.T].mean().item())

n_ts_filter = Qzs2.shape[2]
rates_inf = vae.rnn.observation(Qzs2, v[:, :, :n_ts_filter])

tr_ind=0
m_i = m[tr_ind, :].cpu().numpy().astype(bool)
n_ts = int(np.sum(m_i))
s_i = u[tr_ind, :].cpu().numpy()
vmax_latent = 6

In [ ]:
# get onset of stimuli
st_onset_times = np.argwhere(np.diff(abs(s_i).sum(axis=0))>0).flatten()
print(st_onset_times)

In [ ]:
# reload plot plot_inference_rates_vs_spikes
#import importlib
#import fig_utils.plots
#importlib.reload(fig_utils.plots)
#from fig_utils.plots import plot_inference_rates_vs_spikes, plot_example_input,plot_example_spikes


In [ ]:
m_i_bool = m[tr_ind, :].cpu().numpy().astype(bool)
rates_slice = (
    rates_inf[tr_ind, obs_inds[0] : obs_inds[1], :n_ts, 0].detach().cpu().numpy()
)
spikes_slice = x[tr_ind, :, m_i_bool].cpu().numpy()
spikes_gen_slice = data_gen[tr_ind, :, m_i_bool, 0].cpu().numpy()
latent_slice = Z[tr_ind, :, m_i_bool, 0].cpu().numpy()

plot_inference_rates_vs_spikes(
    rates_slice,
    spikes_slice,
    n_ts=n_ts,
    st_onset_times=st_onset_times,
    n_neurons=x.shape[1],
    save_path="../paper_figures/inference_example.pdf",
    dpi=300,
    show=True,
    n_tics_step=20,
    box_w=1,
    box_h=0.6,
    panel_gap_x=1,
)

plot_example_input(
    s_i,
    m_i_bool,
    n_ts=n_ts,
    st_onset_times=st_onset_times,
    save_path="../paper_figures/example_input.pdf",
    show=True,
    box_w=1,
    box_h=0.6,
)

plot_example_spikes(
    spikes_gen_slice,
    n_ts=n_ts,
    save_path="../paper_figures/example_spikes.pdf",
    show=True,
    box_w=1,
    box_h=1,
    #st_onset_times=st_onset_times,
)

plot_example_latents(
    latent_slice,
    n_ts=n_ts,
    vmax=vmax_latent,
    save_path="../paper_figures/example_latents.pdf",
    show=True,
    box_w=1,
    box_h=0.6,
)

## 3) Stimulus-response (late delay window)


In [ ]:
# Stimulus-response during late delay (length-3 trials, bins 45–65; shared with notebook 13)
units_to_plot = [15, 41, 48]
stim_sess_id = sess_a

act = session_stimulus_delay_activity(vae, task, stim_sess_id, seq_len=3)
boxs_data = stimulus_trial_lists(
    act["r_data"], act["labels"], n_pos=n_pos, n_stim=n_stim, unit_inds=units_to_plot
)
boxs_model = stimulus_trial_lists(
    act["r_model"], act["labels"], n_pos=n_pos, n_stim=n_stim, unit_inds=units_to_plot
)

stim_cmap = mpl.colors.ListedColormap(sns.color_palette("husl", n_colors=n_stim))

for unit_i, unit in enumerate(units_to_plot):
    plot_stimulus_response_unit(
        boxs_data,
        boxs_model,
        unit_i,
        n_pos=n_pos,
        cmap=stim_cmap,
        dpi=300,
        show=True,
        box_w=0.6,
        box_h=0.6,
        panel_gap_x=0.25,
        panel_gap_y=0.15,
        y_max=3,
        save_path="../paper_figures/stimulus_response_unit_" + str(unit) + ".pdf",
    )

## 4) stimulus subspaces and time latents


In [ ]:
n_trials = 120

In [ ]:
import fig_utils.plots
import importlib

importlib.reload(fig_utils.plots)

from fig_utils.plots import (
    plot_basis_2d_subspaces,
    plot_time_latents,
    plot_basis_3d_trajectories,
)

In [ ]:
t_plot

In [ ]:
z_by_pos = [
    position_slice_at_time(z_gen[:n_trials], t_plot, p, n_pcs_time)
    for p in range(n_pos)
]
c_by_pos = [labels_gen[:n_trials, p] for p in range(n_pos)]
plot_basis_2d_subspaces(
    z_by_pos,
    c_by_pos,
    cmap=cmap,
    n_stim=n_stim,
    dpi=300,
    show=True,
    box_w=0.5,
    box_h=0.5,
    s_mean=12,
    s_ind=9,
    alpha_ind=0.2,
    save_path="../paper_figures/basis_2d_subspaces.pdf",
)

plot_time_latents(
    z_gen[:n_trials, :, :70],
    n_pcs_time,
    n_trials=30,
    dpi=300,
    show=True,
    box_w=1,
    box_h=0.6,
    lw=0.5,
    panel_gap_x=0.15,
    panel_gap_y=0.25,
    save_path="../paper_figures/time_latents.pdf",
)

In [ ]:
z_gen.shape

In [ ]:
for pos_ind in range(n_pos):
    plot_basis_3d_trajectories(
        z_gen[:n_trials, :, :70],
        labels_gen[:n_trials],
        pos_ind=pos_ind,
        cmap=cmap,
        n_pcs_time=n_pcs_time,
        save_path="../paper_figures/3d_statespace" + str(pos_ind) + ".png",
        mirror_x=True,
        axis_line_width=5,
        stim_line_width=3,
        window_size=(800, 600),
        zoom=1.2,
    )

## 5) Clamped latents and fixed points



In [ ]:
df_fp = pd.read_pickle("../data/processed/df_fixed_points.pkl")
row_fp = df_fp.loc[df_fp["name"] == model_name]
if row_fp.empty:
    raise ValueError(f"{model_name} not in df_fixed_points.pkl — run notebook 04 first")
row_fp = row_fp.iloc[0]

# Unclamped rollout (for example moving latents) vs clamped (for fixed latents panel)
Z_full = rnn_orth.simulate(u_fp, noise_scale=noise_scale)
Z_frozen = rnn_orth.simulate(
    u_fp,
    noise_scale=noise_scale,
    freeze_indices=[*fixed_inds],
    freeze_time_step=freeze_time_step,
    freeze_noise=freeze_noise,
    freeze="mean",
)

plot_frozen_latents(
    Z_full,
    Z_frozen,
    labels_fp,
    cmap,
    n_pos,
    n_ts_plot_frozen,
    n_pcs_time=n_pcs_time,
    freeze_bin=freeze_time_step,
    bin_size=task_params["bin_size"],
    n_stim=n_stim,
    show=True,
    box_w=1,
    box_h=0.6,
    panel_gap_x=0.15,
    save_path="../paper_figures/frozen_latents.pdf",
)

fp_idx = list(row_fp["freeze_time_steps"]).index(freeze_time_step)
Z_fps = row_fp["fixed_points"][fp_idx]
max_eig_mags = row_fp["max_eig_mags"][fp_idx]
print(f"Stable fixed points: {np.sum(np.array(max_eig_mags) < 1)}")
print(f"Unstable fixed points: {np.sum(np.array(max_eig_mags) > 1)}")

plot_fixed_points_subspaces(
    Z_fps,
    max_eig_mags,
    Z_frozen,
    labels_fp,
    t_decode=freeze_time_step,
    n_pcs_time=n_pcs_time,
    cmap=cmap,
    n_stim=n_stim,
    n_pos=n_pos,
    show=True,
    box_w=0.6,
    box_h=0.6,
    panel_gap_x=0.15,
    save_path="../paper_figures/fixed_points_subspaces.pdf",
)

## 6) Attractor perturbation



In [ ]:
plot_ts = [48, 50, 51, 66]  # , 70]

Z = generate_w_perturb_x(rnn_orth, u=u_gen, noise_scale=noise_scale)

# geometry (same as 07)
pert_space = precompute_perturb_geometry(rnn_orth, n_pcs_time, n_pos)
pert_dir = pert_space[pos]

z1 = n_pcs_time + 2 * pos
z2 = n_pcs_time + 2 * pos + 1
labels_pos = labels_gen[:, pos]

unique_classes = np.unique(labels_pos[labels_pos > -10])
class_means = np.array(
    [
        (Z[labels_pos == c][:, [z1, z2], perturb_at_t]).mean(axis=0)
        for c in unique_classes
    ]
)
pert_z1 = perturbation_goal_bounds(Z[:, z1, perturb_at_t])
pert_z2 = perturbation_goal_bounds(Z[:, z2, perturb_at_t])

goals = np.random.rand(u_gen.shape[0], 2)
goals[:, 0] = goals[:, 0] * (pert_z1[1] - pert_z1[0]) + pert_z1[0]
goals[:, 1] = goals[:, 1] * (pert_z2[1] - pert_z2[0]) + pert_z2[0]

Z_pert = generate_w_perturb_x(
    rnn_orth,
    u=u_gen,
    noise_scale=noise_scale,
    perturb_at_t=perturb_at_t,
    perturb_weights=pert_dir,
    perturb_inds=list(position_latent_indices(n_pcs_time, pos)),
    goal=goals,
    goal_amp=goal_amp,
)

stats = compute_perturbation_distance_stats(
    Z,
    Z_pert,
    labels_pos,
    z1=z1,
    z2=z2,
    t_move_start=perturb_at_t,
    t_move_end=perturb_at_t + 15,
)
display(
    pd.DataFrame(
        [
            {
                k: stats[k]
                for k in stats
                if not hasattr(stats[k], "__len__") or np.ndim(stats[k]) == 0
            }
        ]
    )
)

for highlight_cond in (int(unique_classes[0]), None):
    suffix = "highlight" if highlight_cond is not None else "all"
    plot_perturbation_latent_snapshots_attractor(
        Z,
        Z_pert,
        labels_pos,
        z1=z1,
        z2=z2,
        cmap=cmap,
        plot_ts=plot_ts,
        highlight_cond=highlight_cond,
        bin_size=task_params["bin_size"],
        dpi=300,
        show=True,
        panel_gap_x=0.15,
        save_path=f"../paper_figures/perturbation_snapshots_attractor_{suffix}.pdf",
    )

In [ ]:
plot_distance_moved_vs_class_mean(
    stats["distances_to_manifolds"],
    stats["distance_moved"],
    slope=stats["slope"],
    intercept=stats["intercept"],
    pearson_r=stats["pearson_r"],
    dpi=300,
    show=True,
    box_w=0.6,
    box_h=0.6,
    save_path="../paper_figures/distance_moved_vs_class_mean.pdf",
    max_y=22,
)

## 7) Behavior perturbation demo (one sequence)



In [ ]:
demo_rng_seed = 2

In [ ]:
pos = 0

In [ ]:
rng = np.random.default_rng(demo_rng_seed)
gen_idx = int(rng.choice(np.arange(labels_gen.shape[0])))
valid_target = admissible_targets(labels_gen[gen_idx], pos, n_pos, n_stim)
target = int(rng.choice(valid_target))
u_src = np.repeat(u_gen[gen_idx : gen_idx + 1], 120, axis=0)
labels_highlight = labels_gen[gen_idx]

# goals/geometry (same logic as notebook 07)
goal_means = latent_mean(z_gen, labels_gen, perturb_at_t, n_pcs_time, n_pos, n_stim)
pert_space = precompute_perturb_geometry(rnn_orth, n_pcs_time, n_pos)
pert_dir = pert_space[pos]
z_target_mean = goal_means[pos, target]

Z_unperturbed = generate_w_perturb_x(rnn_orth, u=u_src, noise_scale=noise_scale)
Z_perturbed = generate_w_perturb_x(
    rnn_orth,
    u=u_src,
    noise_scale=noise_scale,
    perturb_at_t=perturb_at_t,
    perturb_weights=pert_dir,
    perturb_inds=list(position_latent_indices(n_pcs_time, pos)),
    goal=z_target_mean,
    goal_amp=goal_amp,
    optogen=0,
)

plot_perturbation_latent_snapshots(
    Z_unperturbed,
    Z_perturbed,
    z_gen,
    labels_gen,
    labels_highlight,
    t_bin=t_snapshot,
    n_pcs_time=n_pcs_time,
    n_pos=n_pos,
    cmap=cmap,
    bin_size=task_params["bin_size"],
    dpi=160,
    show=True,
    save_path="../paper_figures/perturbation_latent_snapshots.pdf",
)
# get response distributions
accs, responses, valid = eval_gen_decoder(
    Z_unperturbed,
    labels_gen,
    delay_ends_gen,
    shared_model,
    response_times,
    n_pos,
    bins_before,
    bins_after,
    return_preds=True,
)
responses = np.apply_along_axis(
    lambda x: np.bincount(x, minlength=n_stim).astype(float), axis=0, arr=responses
).T
responses /= responses.sum(axis=1, keepdims=True)

accs, responses_perturbed, valid = eval_gen_decoder(
    Z_perturbed,
    labels_gen,
    delay_ends_gen,
    shared_model,
    response_times,
    n_pos,
    bins_before,
    bins_after,
    return_preds=True,
)
responses_perturbed = np.apply_along_axis(
    lambda x: np.bincount(x, minlength=n_stim).astype(float),
    axis=0,
    arr=responses_perturbed,
).T
responses_perturbed /= responses_perturbed.sum(axis=1, keepdims=True)

plot_decision_distributions(
    responses,
    responses_perturbed,
    labels_highlight,
    target_cond=target,
    perturb_pos=pos,
    cmap=cmap,
    n_stim=n_stim,
    dpi=160,
    show=True,
    save_path="../paper_figures/decision_distributions.pdf",
)